# K8s RAG — LangChain + Hybrid Search



## 0. Cài đặt

In [1]:
%pip install -q \
    langchain langchain-community langchain-google-genai \
    langchain-huggingface \
    faiss-cpu sentence-transformers rank-bm25 \
    google-generativeai datasets \
    beautifulsoup4 lxml tqdm colorama numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports & Config

In [ ]:
import os, json, re, time, warnings, hashlib
import numpy as np
from bs4 import BeautifulSoup
from datasets import load_dataset
from tqdm.auto import tqdm
from colorama import Fore, Style, init
warnings.filterwarnings('ignore')
init(autoreset=True)

# ── LangChain ──────────────────────────────────────────────────────────────
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# ── Config  ───────────────────────────────────────
CFG = {
    # Dataset
    "dataset_name"    : "mcipriano/stackoverflow-kubernetes-questions",
    "max_samples"     : 5000,
    "min_q_len"       : 30,
    "min_a_len"       : 100,

    # Embedding 
    "embed_model"     : "intfloat/e5-base-v2",

    # Retrieval — Hybrid
    "top_k_dense"     : 10,   # FAISS lấy bao nhiêu trước khi fusion
    "top_k_sparse"    : 10,   # BM25 lấy bao nhiêu trước khi fusion
    "top_k_final"     : 3,    # Số doc đưa vào LLM sau RRF fusion
    "dense_weight"    : 0.5,  # Trọng số FAISS trong EnsembleRetriever
    "sparse_weight"   : 0.5,  # Trọng số BM25 (dense + sparse = 1.0)
    "context_chars"   : 3000,

    # Gemini
    "gemini_api_key"  : os.getenv("GEMINI_API_KEY", "..."), # user tự nhập API
    "gemini_model"    : "gemini-2.5-flash-lite",
    "temperature"     : 0.1,
    "max_tokens"      : 1024,

    # Storage
    "faiss_dir"       : "faiss_k8s_lc",   
    "docs_path"       : "docs_k8s.json",
}

print(f"{Fore.GREEN} Config ready")
print(f"   Hybrid weights: dense={CFG['dense_weight']}, sparse={CFG['sparse_weight']}")
print(f"   top_k: dense={CFG['top_k_dense']}, sparse={CFG['top_k_sparse']}, final={CFG['top_k_final']}")

c:\Users\nhuud\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nhuud\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


 Config ready
   Hybrid weights: dense=0.5, sparse=0.5
   top_k: dense=10, sparse=10, final=3


## 2. Load & Clean Dataset



In [3]:
def strip_html(text):
    if not text: return ""
    soup = BeautifulSoup(text, "lxml")
    for code_tag in soup.find_all('code'):
        code_tag.replace_with(f" `{code_tag.get_text()}` ") # <code>...<code> -> '...'
    for pre in soup.find_all('pre'):
        pre.replace_with(f"\n```\n{pre.get_text().strip()}\n```\n") # <pre>...</pre> -> '''....'''
    text = soup.get_text(separator=" ") # xóa toàn bộ HTML tag như <p>Hello</p><p>World</p> -> Hello World
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">"
             ).replace("&quot;", '"').replace("&#39;", "'") # biến các kí tự đặc biệt trong HTML thành các phép so sánh
    return re.sub(r'\s+', ' ', text).strip()


def is_low_quality(answer: str) -> bool: # trả về True/False nếu answer là rác, ko có ý nghĩa
    a = answer.strip().lower()
    if len(a) < 100: return True
    if a.count('http') > 3 and len(a) < 200: return True
    low_quality_patterns = [
        r'^(try|see|check|look at|refer to|please see)',
        r'^(yes|no|maybe|it depends)\.',
        r'^(this is a duplicate)',
    ]
    return any(re.match(p, a) for p in low_quality_patterns)


def load_data(cfg) -> tuple[list[dict], list[Document]]:
    """
    Load dataset → clean → trả về:
      - raw_docs: list[dict]  (giữ để build BM25)
      - lc_docs:  list[Document]  (để build FAISS qua LangChain)
    """
    print(f"{Fore.YELLOW} Loading dataset...")
    raw   = load_dataset(cfg['dataset_name'], split='train')
    limit = min(cfg['max_samples'], len(raw))
    print(f"   Raw: {len(raw):,} rows → xử lý {limit:,}")

    raw_docs, lc_docs = [], []
    seen   = set()
    skipped = {"short": 0, "low_quality": 0, "duplicate": 0}

    for i, row in enumerate(tqdm(raw.select(range(limit)), desc="Cleaning")):
        q = strip_html(row.get('Question', ''))
        a = strip_html(row.get('Answer', ''))

        if len(q) < cfg['min_q_len'] or len(a) < cfg['min_a_len']:
            skipped['short'] += 1; continue
        if is_low_quality(a):
            skipped['low_quality'] += 1; continue

        q_hash = hashlib.md5(re.sub(r'\s+', ' ', q.lower()).encode()).hexdigest()
        if q_hash in seen:
            skipped['duplicate'] += 1; continue
        seen.add(q_hash)

        doc_id = str(i) # chuyển từ index sang dạng string để lưu biến id 

        # Dict gốc — cần để build BM25 và tra cứu nhanh
        raw_docs.append({
            'id': doc_id,
            'question': q,
            'answer': a,
        })

        # LangChain Document — page_content = Q+A, metadata mang id để join lại
        content = f"Q: {q}\nA: {a}"
        lc_docs.append(Document(
            page_content=content,
            metadata={"id": doc_id, "question": q[:200]},
        ))

    print(f"\n{Fore.GREEN} {len(raw_docs):,} docs sạch")
    print(f"   Skipped — short: {skipped['short']:,} | low_quality: {skipped['low_quality']:,} | dup: {skipped['duplicate']:,}")
    return raw_docs, lc_docs


raw_docs, lc_docs = load_data(CFG)
print(f"\nSample LangChain doc:\n{lc_docs[0].page_content[:300]}...")
print(f"Metadata: {lc_docs[0].metadata}")

 Loading dataset...


   Raw: 30,044 rows → xử lý 5,000


Cleaning: 100%|██████████| 5000/5000 [00:07<00:00, 674.71it/s]



 4,781 docs sạch
   Skipped — short: 126 | low_quality: 64 | dup: 29

Sample LangChain doc:
Q: How to resolve the error no module named pandas when one node (in Airflow's DAG) is successful in using it(pandas) and the other is not? I am unable to deduce as to why I am getting an error no module named pandas. I have checked via `pip3 freeze` and yes, the desired pandas version does show up....
Metadata: {'id': '0', 'question': "How to resolve the error no module named pandas when one node (in Airflow's DAG) is successful in using it(pandas) and the other is not? I am unable to deduce as to why I am getting an error no module"}


## 3. Build Hybrid Index (FAISS + BM25)


In [4]:
def build_or_load_indexes(lc_docs: list[Document], cfg: dict):
    """
    Build FAISS + BM25 indexes.
    FAISS: lưu/load từ disk (tốn thời gian encode).
    BM25:  build nhanh (~vài giây), không cần cache.
    """
    # ── Embedding model ────────────────────────────────────────────────────
    # HuggingFaceEmbeddings wrapper tự xử lý prefix cho E5
    embed_model = HuggingFaceEmbeddings(
        model_name=cfg['embed_model'],
        encode_kwargs={"normalize_embeddings": True,
                       "batch_size": 256},
        model_kwargs={"device": "cpu"},
    )

    # ── FAISS (Dense) ──────────────────────────────────────────────────────
    faiss_dir = cfg['faiss_dir']
    if os.path.exists(faiss_dir):
        print(f"{Fore.CYAN} FAISS index đã có → load từ disk ({faiss_dir})")
        faiss_store = FAISS.load_local(
            faiss_dir,
            embed_model,
            allow_dangerous_deserialization=True,
        )
    else:
        print(f"{Fore.YELLOW} Building FAISS index ({len(lc_docs):,} docs)...")
        faiss_store = FAISS.from_documents(lc_docs, embed_model)
        faiss_store.save_local(faiss_dir)
        print(f"{Fore.GREEN} FAISS saved to {faiss_dir}")

    # ── BM25 (Sparse) ──────────────────────────────────────────────────────
    # BM25Retriever.from_documents() build nhanh — không cần cache
    print(f"{Fore.YELLOW} Building BM25 index...")
    bm25_retriever = BM25Retriever.from_documents(lc_docs)
    bm25_retriever.k = cfg['top_k_sparse']
    print(f"{Fore.GREEN} BM25 ready")

    # ── FAISS retriever wrapper ────────────────────────────────────────────
    faiss_retriever = faiss_store.as_retriever(
        search_kwargs={"k": cfg['top_k_dense']}
    )

    # ── EnsembleRetriever = RRF Fusion ─────────────────────────────────────
    # weights=[dense, sparse] — EnsembleRetriever dùng RRF internally
    hybrid_retriever = EnsembleRetriever(
        retrievers=[faiss_retriever, bm25_retriever],
        weights=[cfg['dense_weight'], cfg['sparse_weight']],
        c=60,    # RRF constant k (default 60 theo paper)
    )

    print(f"\n{Fore.GREEN} Hybrid Retriever ready")
    print(f"   Dense (FAISS):  top_k={cfg['top_k_dense']}, weight={cfg['dense_weight']}")
    print(f"   Sparse (BM25):  top_k={cfg['top_k_sparse']}, weight={cfg['sparse_weight']}")
    print(f"   Fusion method:  RRF (c={60})")

    return hybrid_retriever, faiss_store, bm25_retriever


hybrid_retriever, faiss_store, bm25_retriever = build_or_load_indexes(lc_docs, CFG)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5699.52it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 FAISS index đã có → load từ disk (faiss_k8s_lc)
 Building BM25 index...
 BM25 ready

 Hybrid Retriever ready
   Dense (FAISS):  top_k=10, weight=0.5
   Sparse (BM25):  top_k=10, weight=0.5
   Fusion method:  RRF (c=60)


## 5. LLM & Prompt



In [5]:
SYSTEM = """You are a senior Kubernetes DevOps engineer with 10+ years of production experience.

SCOPE:
- Kubernetes, kubectl, containers, Helm, service mesh, cloud-native systems ONLY.
- OUT of scope: general programming, OS issues unrelated to containers.

BEHAVIOR:
- Think like an engineer debugging a production system under pressure.
- Skip theory — go straight to the fix.
- Assume the user knows basic Kubernetes concepts.
- If multiple causes are possible, pick the most common real-world one.

RULES:
1. If clearly OUT of scope, reply:
   "That's outside my expertise! I specialize in Kubernetes and cloud infrastructure. 😊"
2. Use reference documents as PRIMARY source — only if they are directly relevant to the question.
   If retrieved docs do not match the question, ignore them and answer from standard Kubernetes knowledge.
3. Use general Kubernetes knowledge ONLY when docs are insufficient — never guess.
4. NEVER fabricate kubectl commands, API field names, or YAML keys.
5. Use exact API field names: resources.requests.memory, resources.limits.memory,
   resources.requests.cpu, resources.limits.cpu.
6. If the question is ambiguous, state your assumption in one line before answering.
7. Always prioritize POD-LEVEL debugging first (kubectl logs, describe, events).
   Do NOT jump to cluster-level systems unless explicitly mentioned in the question.
8. Prefer the MOST COMMON real-world fix, not edge cases.
   For CrashLoopBackOff: always start with `kubectl logs <pod> --previous`, not exec or command overrides.
   For HOW-TO: give the single clearest approach, not multiple alternatives.

DETECT QUESTION TYPE FIRST, then apply the matching FORMAT:

TYPE 1 — TROUBLESHOOTING
Triggers: errors, failures, crash, "why is", "not working", "stuck", CrashLoopBackOff/Pending/OOMKilled.
Format:
Root Cause:
<ONE most common pod/container-level cause>

Solution:
- <step 1 with exact command>
- <step 2 with exact command>
- <step 3 with exact command — max 3>

Verification:
<ONE read-only kubectl command — no prose>

TYPE 2 — HOW-TO
Triggers: "how do I", "how to", "how can I", "steps to".
Format:
Steps:
- <step 1 with exact command>
- <step 2 with exact command>
- <step 3 with exact command — max 3>

Verification:
<ONE read-only kubectl command — no prose>

TYPE 3 — CONCEPT
Triggers: "what is", "what are", "explain", "difference between", "when to use".
Format:
<2-3 sentences max — practical definition, real-world usage, no textbook phrasing>

STYLE:
- Concise, direct, zero fluff.
- Real commands with real names — not <placeholder> style when avoidable.
- No filler phrases ("Great question!", "Sure!", "Of course!").
- If YAML is needed, write it as a short inline block — never use heredoc (<<EOF) in steps.
"""

# ── LLM ────────────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model=CFG['gemini_model'],
    google_api_key=CFG['gemini_api_key'],
    temperature=CFG['temperature'],
    max_output_tokens=CFG['max_tokens'],
)

# ── Prompt template ─────────────────────────────────────────────────────────
# LangChain ChatPromptTemplate — system + human message
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Question: {question}\n\n"
      "Additional context from our knowledge base (use only if relevant):\n{context}"),
])

print(f"{Fore.GREEN} LLM & Prompt ready")
print(f"   Model: {CFG['gemini_model']}  |  temp={CFG['temperature']}")

 LLM & Prompt ready
   Model: gemini-2.5-flash-lite  |  temp=0.1


## 6. Context Formatter

In [6]:
def format_context(docs: list[Document], ctx_chars: int = CFG['context_chars']) -> str:
    if not docs:
        return "NO_RELEVANT_DOCS"

    parts = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        if len(content) > ctx_chars:
            content = content[:ctx_chars] + "..."
        q_preview = doc.metadata.get('question', '')[:200]
        parts.append(f"[Doc {i}]\n{content}")

    return "\n\n---\n\n".join(parts)

## 7. Build RAG Chain (LCEL)


In [7]:
rag_chain = (
    RunnablePassthrough.assign(
        # 1. Retrieve documents
        retrieved_docs=RunnableLambda(
            lambda x: hybrid_retriever.invoke(x["question"])[:CFG['top_k_final']]
        ),
    )
    | RunnablePassthrough.assign(
        # 2. Format thành context string
        context=RunnableLambda(
            lambda x: format_context(x["retrieved_docs"])
        )
    )
    | {
        # 3. Build prompt inputs
        "answer"  : prompt | llm | StrOutputParser(),
        "docs"    : lambda x: x["retrieved_docs"],
        "question": lambda x: x["question"],
    }
)

print(f"{Fore.GREEN} RAG chain built (LCEL)")
print("Chain: HybridRetriever → format_context → Prompt → Gemini → StrOutput")

 RAG chain built (LCEL)
Chain: HybridRetriever → format_context → Prompt → Gemini → StrOutput


## 8. Answer Helper 

In [8]:
def answer(question: str) -> dict:
    t0 = time.time()

    result = rag_chain.invoke({"question": question})
    docs   = result["docs"]
    ans    = result["answer"]

    print(f"\n{Fore.CYAN}{'━'*60}")
    print(f"{Fore.WHITE} {question}")
    if docs:
        print(f"{Fore.YELLOW} {len(docs)} docs retrieved (Hybrid RRF)")
        for i, d in enumerate(docs, 1):
            q_preview = d.metadata.get('question', d.page_content[:80])
            print(f"   [{i}] {q_preview[:90]}")
    else:
        print(f"{Fore.YELLOW} Không có doc liên quan")
    print(f"{Fore.CYAN}{'─'*60}")
    print(f"{Fore.GREEN}{ans}")

    elapsed = time.time() - t0
    print(f"\n{Style.RESET_ALL}{Fore.CYAN}{'─'*60}")
    print(f"{Fore.YELLOW}  {elapsed:.1f}s")
    print(f"{Fore.CYAN}{'━'*60}")

    return {
        "question" : question,
        "answer"   : ans,
        "n_sources": len(docs),
        "latency_s": round(elapsed, 1),
    }


print(f"{Fore.GREEN} Ready — gọi answer('câu hỏi của bạn')")

 Ready — gọi answer('câu hỏi của bạn')


## 9. Test 

In [9]:
answer("What is a pod in Kubernetes?")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 What is a pod in Kubernetes?
 3 docs retrieved (Hybrid RRF)
   [1] In docker host and the containers do have separate process name-space. In case of Kubernet
   [2] What is the difference between using Kustomize and Tekton for deployment? To me it looks l
   [3] I see that kubernets uses pod and then in each pod there can be multiple containers. Examp
────────────────────────────────────────────────────────────
TYPE 3 — CONCEPT
A Pod is the smallest deployable unit in Kubernetes, representing a single instance of a running process in your cluster. It's a wrapper for one or more containers that share storage, network resources, and specifications about how to run. Pods are not VMs; containers within a Pod share the host's kernel and can communicate via localhost.

────────────────────────────────────────────────────────────
  1.8s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


{'question': 'What is a pod in Kubernetes?',
 'answer': "TYPE 3 — CONCEPT\nA Pod is the smallest deployable unit in Kubernetes, representing a single instance of a running process in your cluster. It's a wrapper for one or more containers that share storage, network resources, and specifications about how to run. Pods are not VMs; containers within a Pod share the host's kernel and can communicate via localhost.",
 'n_sources': 3,
 'latency_s': 1.8}

In [10]:
answer("My pod is stuck in CrashLoopBackOff. How do I debug it?")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 My pod is stuck in CrashLoopBackOff. How do I debug it?
 3 docs retrieved (Hybrid RRF)
   [1] Kubernetes Pods are stuck with a STATUS of `Terminating` after the Deployment (and Service
   [2] In Kubernetes, when a Pod repeatedly crashes and is in `CrashLoopBackOff` status, it is no
   [3] I am trying to follow the instructions on this webpage to debug an app deployed to Kuberne
────────────────────────────────────────────────────────────
Root Cause:
The container is crashing immediately after starting, preventing it from staying alive long enough to be debugged.

Solution:
- `kubectl logs <pod-name> --previous`
- `kubectl describe pod <pod-name>`
- `kubectl exec -it <pod-name> -- /bin/sh` (if the container is running long enough to exec into)

Verification:
`kubectl get pods`

────────────────────────────────────────────────────────────
  1.5s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


{'question': 'My pod is stuck in CrashLoopBackOff. How do I debug it?',
 'answer': 'Root Cause:\nThe container is crashing immediately after starting, preventing it from staying alive long enough to be debugged.\n\nSolution:\n- `kubectl logs <pod-name> --previous`\n- `kubectl describe pod <pod-name>`\n- `kubectl exec -it <pod-name> -- /bin/sh` (if the container is running long enough to exec into)\n\nVerification:\n`kubectl get pods`',
 'n_sources': 3,
 'latency_s': 1.5}

## 10. Interactive Chat

In [ ]:
print(f"{Fore.YELLOW}")
print("╔══════════════════════════════════════════════╗")
print("║  🤖  K8s RAG  —  LangChain + Hybrid Search   ║")
print("║  BM25 + E5 Dense → RRF → Gemini              ║")
print("║  Gõ câu hỏi  |  'exit' để thoát             ║")
print("╚══════════════════════════════════════════════╝")
print(Style.RESET_ALL)

while True:
    try:
        q = input(f"{Fore.CYAN}You: {Style.RESET_ALL}").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not q: continue
    if q.lower() in ['exit', 'quit', 'q']:
        print(f"{Fore.YELLOW}👋 Bye!")
        break
    answer(q)